**Expense Manager Agent**

In [ ]:
# loading api key  
import os 
from dotenv import load_dotenv
load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")



In [8]:
# model creation
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    },
    temperature=0.1
)

In [13]:
# tool creation 1
from langchain_core.tools import tool

@tool
def calculate_discount(price: float, discount_percent: float) -> dict:
    """Calculate discount and final price."""
    
    discount = price * discount_percent / 100
    final_price = price - discount

    return {
        "original_price": price,
        "discount": discount,
        "final_price": final_price
    }
    

In [14]:
# tool creation 2
@tool
def split_bill(total_amount: float, number_of_people: int) -> dict:
    """Divide a bill equally among multiple people."""

    amount_per_person = total_amount / number_of_people

    return {
        "total_amount": total_amount,
        "number_of_people": number_of_people,
        "amount_per_person": amount_per_person
    }

In [15]:
# creating agent
from langchain.agents import create_agent
agent = create_agent(
    model = llm,
    tools = [calculate_discount, split_bill]
)

In [16]:
user_question = "A product costs ₹5000 and has a 20% discount. What is the final price?"

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_question
        }
    ]
})

print(result["messages"][-1].content)

The product’s original price is ₹5,000.  
With a 20 % discount:

- Discount amount = 20 % × ₹5,000 = ₹1,000  
- Final price = ₹5,000 – ₹1,000 = **₹4,000**


In [17]:
user_question = "Split a bill of ₹3000 among 4 people."

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_question
        }
    ]
})

print(result["messages"][-1].content)

Each person should pay ₹750.
